# Fraud Detection Hackathon – Categorical EDA (Percentages)

This notebook mirrors the categorical exploratory analysis, but plots all bar charts on a percentage scale without line plots.

## Objectives
- Reuse the cleaning pipeline for categorical variables.
- Display side-by-side bar charts where the y-axis shows the share of transactions per label.
- Omit line charts while preserving mutual-information scoring.

In [1]:
# Imports and global configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

In [2]:
# Load training data
DATA_PATH = "data/train.csv"
raw_df = pd.read_csv(DATA_PATH)

print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())

Raw shape: (195401, 16)


,order_id,order_date,amount_usd,payment_method,billing_address_country_code,shipping_address_country_code,avs_code,email_amount_spent_avg_180d,email_orders_count_180d,session_visitor_type,shipping_method_type,card_expiration_date,bin_brand,bin_type,label,suspicious_activity_score
0,181852054,2022-06-02 15:28:13,611.03,payment_method_2,SA,SA,avs_code_8,NaN,NaN,returning,E,05/2025,mastercard,credit,0.0,7.801596
1,181265052,2022-05-21 15:53:20,334.44,payment_method_1,FR,FR,avs_code_1,591.141667,6.0,returning,S,NaN,NaN,NaN,0.0,8.117476
2,149633145,2021-09-30 09:05:06,611.85,payment_method_1,HK,HK,avs_code_1,NaN,NaN,returning,E;E,NaN,NaN,NaN,0.0,6.240885
3,149158895,2021-09-22 15:19:15,198.18,payment_method_2,RU,RU,avs_code_5,473.027368,38.0,returning,E,03/2023,visa,debit,0.0,5.188745
4,169664695,2022-02-18 08:59:32,434.71,payment_method_3,HK,HK,avs_code_5,NaN,NaN,returning,E,NaN,NaN,NaN,0.0,5.738602


## Data Cleaning Strategy
The same cleaning steps are applied as in the count-based notebook to guarantee comparability:

1. Drop duplicate `order_id` rows.
2. Remove rows with missing labels or unparseable `order_date` values.
3. Fill categorical NaNs with `'Unknown'` and numeric NaNs with the column median.

In [3]:
# Apply cleaning strategy
clean_df = raw_df.copy()

initial_rows = len(clean_df)
clean_df = clean_df.drop_duplicates(subset="order_id")
duplicates_removed = initial_rows - len(clean_df)

label_missing = clean_df["label"].isna().sum()
if label_missing:
    clean_df = clean_df.dropna(subset=["label"])
    print(f"Dropped {label_missing} rows with missing label.")

clean_df["order_date"] = pd.to_datetime(clean_df["order_date"], errors="coerce")
invalid_dates = clean_df["order_date"].isna().sum()
if invalid_dates:
    clean_df = clean_df.dropna(subset=["order_date"])
    print(f"Dropped {invalid_dates} rows where `order_date` could not be parsed.")

categorical_cols = clean_df.select_dtypes(include="object").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "order_id"]

numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "label"]

for col in categorical_cols:
    if clean_df[col].isna().any():
        clean_df[col] = clean_df[col].fillna("Unknown")

median_imputations = {}
for col in numeric_cols:
    if clean_df[col].isna().any():
        median_value = clean_df[col].median()
        clean_df[col] = clean_df[col].fillna(median_value)
        median_imputations[col] = median_value

print(f"Rows removed as duplicates: {duplicates_removed}")
print(f"Final cleaned shape: {clean_df.shape}")
print("Numeric columns imputed with median:", median_imputations if median_imputations else "None")

missing_summary = clean_df.isna().sum()
if missing_summary.sum() == 0:
    print("Remaining missing values per column: None")
else:
    display(missing_summary[missing_summary > 0])

Dropped 5 rows with missing label.
Rows removed as duplicates: 5135
Final cleaned shape: (190261, 16)
Numeric columns imputed with median: {'email_amount_spent_avg_180d': np.float64(459.94375), 'email_orders_count_180d': np.float64(4.0)}
Remaining missing values per column: None


In [4]:
# Identify categorical features (excluding order_id and temporal fields)
excluded_categoricals = {"order_id"}
if "order_date" in clean_df.columns:
    excluded_categoricals.add("order_date")

categorical_features = [
    col for col in clean_df.columns
    if clean_df[col].dtype == "object" and col not in excluded_categoricals
]

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

summary_rows = []
for col in categorical_features:
    value_counts = clean_df[col].value_counts()
    summary_rows.append({
        "feature": col,
        "unique_values": value_counts.shape[0],
        "top_category": value_counts.index[0],
        "top_category_pct": (value_counts.iloc[0] / len(clean_df)) * 100
    })

categorical_summary = pd.DataFrame(summary_rows).sort_values("unique_values")
display(categorical_summary)

Categorical features (9): ['payment_method', 'billing_address_country_code', 'shipping_address_country_code', 'avs_code', 'session_visitor_type', 'shipping_method_type', 'card_expiration_date', 'bin_brand', 'bin_type']


,feature,unique_values,top_category,top_category_pct
4,session_visitor_type,3,returning,70.734412
8,bin_type,4,credit,42.203079
0,payment_method,9,payment_method_2,69.113481
7,bin_brand,11,visa,33.970703
3,avs_code,25,avs_code_4,24.038558
5,shipping_method_type,156,E,46.003648
2,shipping_address_country_code,167,GB,15.426703
1,billing_address_country_code,167,GB,15.208582
6,card_expiration_date,188,Unknown,30.981126


In [ ]:
# Helper utilities for percentage-based plots
PALETTE = {'Label 0': "#1f77b4", 'Label 1': "#ff7f0e"}

def summarise_categorical_feature(dataframe, feature, top_n=12):
    working = dataframe[[feature, "label"]].copy()
    working[feature] = working[feature].fillna("Unknown")
    top_categories = working[feature].value_counts().nlargest(top_n).index
    working.loc[~working[feature].isin(top_categories), feature] = "Other"
    working[feature] = working[feature].astype(str)

    overall_counts = (
        working[feature]
        .value_counts()
        .rename_axis(feature)
        .reset_index(name="total_count")
    )

    label_counts = (
        working.groupby([feature, "label"])
        .size()
        .reset_index(name="count")
        .pivot(index=feature, columns="label", values="count")
        .fillna(0)
        .astype(int)
    )

    for label_value in [0, 1]:
        if label_value not in label_counts.columns:
            label_counts[label_value] = 0

    label_counts = label_counts[[0, 1]].reindex(overall_counts[feature])
    label_counts["overall"] = label_counts[0] + label_counts[1]

    total_count_all_categories = label_counts["overall"].sum()
    percentage_table = pd.DataFrame({
        "label_0_pct": (label_counts[0] / label_counts["overall"]) * 100,
        "label_1_pct": (label_counts[1] / label_counts["overall"]) * 100,
        "overall_pct": (label_counts["overall"] / total_count_all_categories) * 100
    })

    summary_table = overall_counts.merge(
        label_counts.reset_index(), on=feature, how="left"
    ).merge(
        percentage_table.reset_index(), on=feature, how="left"
    )

    return summary_table, percentage_table

def plot_categorical_feature(dataframe, feature, top_n=12):
    summary_table, percentage_table = summarise_categorical_feature(dataframe, feature, top_n=top_n)
    category_order = summary_table[feature].tolist()

    print(f"\nFeature: {feature}")
    display(summary_table)

    plotting_df = percentage_table.reset_index().melt(
        id_vars=feature,
        value_vars=["label_0_pct", "label_1_pct"],
        var_name="label",
        value_name="percentage"
    )
    plotting_df["label"] = plotting_df["label"].map({
        "label_0_pct": "Label 0",
        "label_1_pct": "Label 1"
    })

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=plotting_df,
        x=feature,
        y="percentage",
        hue="label",
        hue_order=["Label 0", "Label 1"],
        palette=PALETTE,
        order=category_order
    )
    plt.title(f"Percentage distribution by label – {feature}")
    plt.xlabel(feature)
    plt.ylabel('percentage')
    plt.xticks(rotation=45)
    plt.ylim(0, 100)
    plt.legend(title="Label")
    plt.tight_layout()
    plt.show()

In [6]:
# Generate percentage-based visuals for every categorical feature
for feature in categorical_features:
    plot_categorical_feature(clean_df, feature, top_n=12)

NameError: name 'total_count_all_categories' is not defined

In [ ]:
# Mutual information scores (unchanged)
encoded_df = pd.DataFrame({
    col: clean_df[col].astype("category").cat.codes
    for col in categorical_features
})

mi_scores = mutual_info_classif(
    encoded_df,
    clean_df["label"].astype(int),
    discrete_features=True,
    random_state=42,
)

mi_report = (
    pd.DataFrame({
        "feature": categorical_features,
        "mutual_information": mi_scores,
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

print("Mutual information (higher is more informative about the fraud label):")
display(mi_report)

## Notes
- Percentages for each category sum to 100% across the two labels.
- The `overall_pct` column in the summary tables indicates the share of total transactions represented by each category.
- Consider adjusting `top_n` if you need more or fewer categories per chart.